# Milestone 6E: Wide-Column NoSQL Data Modeling (Delta Lake & Cassandra Emulation)

This notebook explores the **wide-column data model** tailored for high-volume AML financial analytics.

### Key Concepts:
- **Cassandra CQL Schema Design**: Partition keys for uniform distribution and clustering keys for fast time-series slices.
- **Delta Lake Physical Storage**: Emulating wide-column semantics on Databricks Community Edition.
- **Schema Evolution**: Adding new feature columns dynamically without full table rewrites.
- **ACID Upserts**: High-throughput profile updates using `MERGE INTO`.
- **Time Travel**: Historical audit logs (`VERSION AS OF`) for regulatory compliance.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.wide_column_model import create_spark_session, WideColumnStore

spark = create_spark_session('Databricks-M6E-WideColumn')
store = WideColumnStore(spark, base_storage_dir='../outputs/wide_column_store')
print('Initialized Wide-Column storage engine format:', store.format)


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


2026-10-29 12:06:20,608 [INFO] src.wide_column_model — Storage engine initialized using format='parquet' at 'C:\Projects\Mule-tiplier Extended\outputs\wide_column_store'


Initialized Wide-Column storage engine format: parquet


## 1. Inspect Cassandra Production CQL Design
Read the production Cassandra CQL schema specification ([`docs/cassandra_schema.cql`](../docs/cassandra_schema.cql)).


In [2]:
with open('../docs/cassandra_schema.cql', 'r') as f:
    cql_text = f.read()
print('Total CQL lines:', len(cql_text.splitlines()))
print('\n'.join(cql_text.splitlines()[:35]))


Total CQL lines: 232
-- ============================================================================
-- Chain Reaction: Money Mule Detection â€” Cassandra Wide-Column Data Model
-- ============================================================================
-- Design Architecture Document: Production NoSQL Cassandra CQL Schema
-- Target Workload: High-throughput ingestion, low-latency AML account lookups,
-- time-series transaction event queries, and audit-ready risk scoring.
-- ============================================================================

-- ----------------------------------------------------------------------------
-- 1. KEYSPACE DEFINITION
-- ----------------------------------------------------------------------------
-- In production, NetworkTopologyStrategy distributes replicas across multiple
-- availability zones or data centers. For local/development environments,
-- SimpleStrategy with replication_factor 3 is standard.
CREATE KEYSPACE IF NOT EXISTS aml
WITH re

## 2. Ingest Sample Data into Wide-Column Delta Tables
Create physical tables for `transaction_events` (partitioned by date) and `account_profiles` (wide row).


In [3]:
tx_path = '../data/processed/processed_df.csv'
feat_path = '../data/processed/account_features.csv'

tx_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(tx_path).limit(10000)
feat_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(feat_path).limit(5000)

paths = store.create_tables(tx_df, feat_df)
print('Physical storage paths created:', paths)


2026-10-29 12:06:26,424 [INFO] src.wide_column_model — Persisting Wide-Column tables...


2026-10-29 12:06:30,753 [INFO] src.wide_column_model — Tables created and registered: aml_transaction_events, aml_account_profiles


Physical storage paths created: {'transaction_events': 'C:\\Projects\\Mule-tiplier Extended\\outputs\\wide_column_store\\transaction_events', 'account_profiles': 'C:\\Projects\\Mule-tiplier Extended\\outputs\\wide_column_store\\account_profiles'}


## 3. Schema Evolution (Adding Dynamic Feature Columns)
Dynamically attach new ML risk scores to the wide-row profiles.


In [4]:
import pyspark.sql.functions as F
new_scores = feat_df.select('account').withColumn('spark_gbt_risk_score', F.rand() * 0.9)
evolved = store.demonstrate_schema_evolution(new_scores)
evolved.select('account', 'in_degree', 'total_in', 'spark_gbt_risk_score').show(5)


2026-10-29 12:06:30,784 [INFO] src.wide_column_model — Executing Wide-Column Schema Evolution...


2026-10-29 12:06:32,155 [INFO] src.wide_column_model — Schema evolved successfully. Current column count: 13


+---------+---------+------------------+--------------------+
|  account|in_degree|          total_in|spark_gbt_risk_score|
+---------+---------+------------------+--------------------+
|100428660|   1084.0|         479594.79|  0.6169985256136737|
|1004286A8|    653.0|       271081.3716|  0.7582731701280147|
|1004286F0|    108.0|35550.743200000004|0.013202706188212466|
|100428738|     98.0|      42239.402615|  0.1423291325080792|
|100428780|    117.0|126850.67940000001|  0.1412283239998682|
+---------+---------+------------------+--------------------+
only showing top 5 rows


## 4. Cassandra Query Emulation
Simulate single-row lookup, ego-subgraph partition scans, and materialized view queries.


In [5]:
sample_account = feat_df.first()['account']
queries = store.simulate_cassandra_queries(sample_account)
print(f'Account Profile for {sample_account}:')
queries['single_account_lookup'].show(1)
spark.stop()


2026-10-29 12:06:32,358 [INFO] src.wide_column_model — Executing Cassandra query emulation for account '100428660'...


Account Profile for 100428660:


+---------+---------+---------+----------+-----------------+--------------------+----------------+------------------------+---------------------+-------------+--------------------+--------------------+--------------------+
|  account|in_degree| total_in|out_degree|        total_out|            pagerank|avg_holding_time|has_outflow_after_inflow|unique_counterparties|is_laundering|        last_updated| last_10_txn_amounts|spark_gbt_risk_score|
+---------+---------+---------+----------+-----------------+--------------------+----------------+------------------------+---------------------+-------------+--------------------+--------------------+--------------------+
|100428660|   1084.0|479594.79|  168672.0|5.276229120975E10|1.564552047694313...|             0.0|                       1|                14775|            1|2026-10-29 12:06:...|[100.0, 250.0, 50.0]|  0.6169985256136737|
+---------+---------+---------+----------+-----------------+--------------------+----------------+----------